In [1]:
import pandas as pd

print("Pandas is installed successfully!")
print("Version:", pd.__version__)

Pandas is installed successfully!
Version: 2.3.3


In [2]:
# Import the libraries we need
import requests        # for making API calls
import pandas as pd    # for data manipulation
import time            # to pause between API calls (respect rate limits)
import os              # for file path operations

# Confirm everything imported correctly
print("Libraries loaded successfully")
print(f"pandas version: {pd.__version__}")

Libraries loaded successfully
pandas version: 2.3.3


## CoinGecko API Endpoint

We will use the CoinGecko Markets endpoint to retrieve cryptocurrency market data.

Endpoint:

https://api.coingecko.com/api/v3/coins/markets

Parameters:
- vs_currency: Currency for prices (e.g., usd)
- order: Sort order for results
- per_page: Number of coins to return
- page: Page number
- sparkline: Include sparkline data (true/false)

The endpoint returns:
- Coin name
- Symbol
- Current price
- Market capitalization
- Trading volume
- Price change percentages

In [3]:
#Function Definitio
def get_coin_data(coin_id, days=365):
    """
    Pull daily price and volume data from CoinGecko.
    coin_id: "bitcoin" or "ethereum"
    days: number of days of history to pull
    Returns a pandas DataFrame
    """
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart"

    params = {
        "vs_currency": "usd",
        "days": days,
        "interval": "daily"
    }

    # Make the API request
    response = requests.get(url, params=params)

    # Check if the request was successful
    if response.status_code != 200:
        print(f"Error: {response.status_code} - {response.text}")
        return None

    # Parse the JSON response
    data = response.json()

    # Extract prices and volumes
    prices = data["prices"]            # list of [timestamp, price]
    volumes = data["total_volumes"]    # list of [timestamp, volume]

    # Convert to DataFrame
    df_price = pd.DataFrame(prices, columns=["timestamp", f"{coin_id}_price_usd"])
    df_volume = pd.DataFrame(volumes, columns=["timestamp", f"{coin_id}_volume_usd"])

    # Merge price and volume on timestamp
    df = pd.merge(df_price, df_volume, on="timestamp")

    # Convert timestamp from milliseconds to readable date
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms").dt.date
    df = df.drop("timestamp", axis=1)

    # Put date first
    df = df[["date", f"{coin_id}_price_usd", f"{coin_id}_volume_usd"]]

    #Enforce one row per date
    df = df.groupby("date", as_index=False).last()

    return df

print("Function defined. Ready to pull data.")

Function defined. Ready to pull data.


In [4]:
# Test: pull 7 days of Bitcoin data ONLY
df_test = get_coin_data("bitcoin", days=7)

print(df_test)
print(f"\nShape: {df_test.shape}")
print(f"Columns: {df_test.columns.tolist()}")

         date  bitcoin_price_usd  bitcoin_volume_usd
0  2026-06-14       64377.576689        1.752280e+10
1  2026-06-15       65713.623560        2.288758e+10
2  2026-06-16       66300.958537        3.391518e+10
3  2026-06-17       65598.943967        2.556770e+10
4  2026-06-18       64422.573665        3.234518e+10
5  2026-06-19       62900.227093        3.127974e+10
6  2026-06-20       63588.905843        2.025176e+10

Shape: (7, 3)
Columns: ['date', 'bitcoin_price_usd', 'bitcoin_volume_usd']


In [5]:
# Pull Bitcoin data — 365 days
print("Pulling Bitcoin data...")
df_btc = get_coin_data("bitcoin", days=365)
print(f"Bitcoin: {df_btc.shape[0]} rows pulled")

# Wait (rate limit safety)
print("Waiting 12 seconds before next call...")
time.sleep(12)

# Pull Ethereum data — 365 days
print("Pulling Ethereum data...")
df_eth = get_coin_data("ethereum", days=365)
print(f"Ethereum: {df_eth.shape[0]} rows pulled")

# Wait again (important — don't skip this)
print("Waiting 12 seconds before next call...")
time.sleep(12)

# Pull USDT data — 365 days
print("Pulling USDT data...")
df_usdt = get_coin_data("tether", days=365)
print(f"USDT: {df_usdt.shape[0]} rows pulled")

# Merge step-by-step (safer than one big merge chain)
df_crypto = pd.merge(df_btc, df_eth, on="date", how="inner")
df_crypto = pd.merge(df_crypto, df_usdt, on="date", how="inner")

# Final checks
print(f"\nMerged dataset shape: {df_crypto.shape}")
print(f"Date range: {df_crypto['date'].min()} to {df_crypto['date'].max()}")

print("\nColumns:")
print(df_crypto.columns.tolist())

print("\nFirst 3 rows:")
print(df_crypto.head(3))

Pulling Bitcoin data...
Bitcoin: 365 rows pulled
Waiting 12 seconds before next call...
Pulling Ethereum data...
Ethereum: 365 rows pulled
Waiting 12 seconds before next call...
Pulling USDT data...
USDT: 365 rows pulled

Merged dataset shape: (365, 7)
Date range: 2025-06-21 to 2026-06-20

Columns:
['date', 'bitcoin_price_usd', 'bitcoin_volume_usd', 'ethereum_price_usd', 'ethereum_volume_usd', 'tether_price_usd', 'tether_volume_usd']

First 3 rows:
         date  bitcoin_price_usd  bitcoin_volume_usd  ethereum_price_usd  \
0  2025-06-21      103290.105145        3.063275e+10         2405.695434   
1  2025-06-22      101532.568385        2.164446e+10         2270.581996   
2  2025-06-23      100852.582646        5.128714e+10         2227.428727   

   ethereum_volume_usd  tether_price_usd  tether_volume_usd  
0         2.048282e+10          1.000017       2.772718e+10  
1         1.615561e+10          1.000177       4.036210e+10  
2         2.818144e+10          1.000176       3.761731e

In [6]:
df_crypto.duplicated(subset=["date"]).sum()

np.int64(0)

In [7]:
df_crypto[df_crypto.duplicated(subset=["date"], keep=False)].sort_values("date")

,date,bitcoin_price_usd,bitcoin_volume_usd,ethereum_price_usd,ethereum_volume_usd,tether_price_usd,tether_volume_usd


In [8]:
#Bitcoin
df_btc = get_coin_data("bitcoin", days=365)
df_btc["date"].is_unique

True

In [9]:
# Ethereum
df_eth = get_coin_data("ethereum", days=365)
print(df_eth["date"].is_unique)

Error: 429 - {"status":{"error_code":429,"error_message":"You've exceeded the Rate Limit. Please visit https://www.coingecko.com/en/api/pricing to subscribe to our API plans for higher rate limits."}}


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# USDT (Tether)
df_usdt = get_coin_data("tether", days=365)
print(df_usdt["date"].is_unique)
print(df_usdt.shape)

In [ ]:
# Merge Ethereum + USDT on date (inner join keeps only matching dates)
df_crypto = df_eth.merge(df_usdt, on="date", how="inner")

print(df_crypto.shape)
print(df_crypto.columns)

In [ ]:
# Merge BTC
df_btc = get_coin_data("bitcoin", days=365)

print(df_btc.shape)
print(df_btc.columns)
print(df_btc["date"].is_unique)

In [ ]:
df_btc.shape, df_btc.columns, df_btc["date"].is_unique

In [ ]:
# Start with BTC as base (best anchor)
df_crypto = df_btc.merge(df_eth, on="date", how="inner")

# Add USDT liquidity layer
df_crypto = df_crypto.merge(df_usdt, on="date", how="inner")

# Final structure check
print("Shape:", df_crypto.shape)
print(df_crypto.columns)
print("Duplicate dates:", df_crypto["date"].duplicated().sum())

In [ ]:
# Total market liquidity proxy (activity in crypto ecosystem)
df_crypto["market_liquidity"] = (
    df_crypto["bitcoin_volume_usd"] +
    df_crypto["ethereum_volume_usd"] +
    df_crypto["tether_volume_usd"]
)

print(df_crypto[["date", "market_liquidity"]].head())

In [ ]:
df_crypto["liquidity_index"] = (
    df_crypto["market_liquidity"] /
    df_crypto["market_liquidity"].rolling(7, min_periods=1).mean()
)

print(df_crypto[["date", "liquidity_index"]].head(10))

In [ ]:
# 1. Recompute returns (daily market movement)
df_crypto["btc_return"] = df_crypto["bitcoin_price_usd"].pct_change()
df_crypto["eth_return"] = df_crypto["ethereum_price_usd"].pct_change()

# Fill first NaN from returns
df_crypto["btc_return"] = df_crypto["btc_return"].fillna(0)
df_crypto["eth_return"] = df_crypto["eth_return"].fillna(0)

# 2. Smooth signals (light smoothing)
df_crypto["btc_signal"] = df_crypto["btc_return"].rolling(3, min_periods=1).mean()
df_crypto["eth_signal"] = df_crypto["eth_return"].rolling(3, min_periods=1).mean()

# 3. Raw Crypto Market Index (CMI)
df_crypto["CMI"] = (
    0.45 * df_crypto["btc_signal"] +
    0.35 * df_crypto["eth_signal"] +
    0.20 * df_crypto["liquidity_index"]
)

# 4. Handle missing values
df_crypto["CMI"] = df_crypto["CMI"].bfill()

# 5. Normalize to 0–100 scale (IMPORTANT for MEPS later)
df_crypto["CMI_index"] = (
    (df_crypto["CMI"] - df_crypto["CMI"].min()) /
    (df_crypto["CMI"].max() - df_crypto["CMI"].min())
) * 100

# 6. Final preview
print(df_crypto[["date", "CMI", "CMI_index"]].head(10))